In [ ]:
# main.py - единая точка входа
import os
import json
import hashlib
from pathlib import Path
from typing import List, Dict, Optional
import sqlite3
from datetime import datetime

# Установка: pip install anthropic requests pillow pypdf2 python-docx python-pptx opencv-python pandas matplotlib seaborn scikit-learn

import anthropic
import requests
from PIL import Image
import PyPDF2
import docx
from pptx import Presentation
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ======================== КОНФИГУРАЦИЯ ========================
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-key-here")
DB_PATH = "educational_materials.db"
DOWNLOAD_DIR = Path("downloads")
DOWNLOAD_DIR.mkdir(exist_ok=True)

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-sonnet-4-20250514"

# ======================== БАЗА ДАННЫХ ========================
def init_db():
    """Создание схемы БД с дедупликацией"""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    
    c.execute('''CREATE TABLE IF NOT EXISTS materials (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        url_hash TEXT UNIQUE NOT NULL,
        url TEXT NOT NULL,
        subject TEXT,
        topic TEXT,
        text_content TEXT,
        annotation TEXT,
        moderation_result TEXT,
        is_approved INTEGER,
        media_descriptions TEXT,
        prev_material_id INTEGER,
        next_material_id INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (prev_material_id) REFERENCES materials(id),
        FOREIGN KEY (next_material_id) REFERENCES materials(id)
    )''')
    
    c.execute('''CREATE TABLE IF NOT EXISTS media_files (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        material_id INTEGER,
        file_path TEXT,
        media_type TEXT,
        description TEXT,
        FOREIGN KEY (material_id) REFERENCES materials(id)
    )''')
    
    conn.commit()
    conn.close()

def get_url_hash(url: str) -> str:
    """Хеш для дедупликации"""
    return hashlib.sha256(url.encode()).hexdigest()

def material_exists(url: str) -> Optional[int]:
    """Проверка существования материала"""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT id FROM materials WHERE url_hash = ?", (get_url_hash(url),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

def save_material(data: Dict) -> int:
    """Сохранение/обновление материала"""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    
    url_hash = get_url_hash(data['url'])
    existing_id = material_exists(data['url'])
    
    if existing_id:
        c.execute('''UPDATE materials SET 
            subject=?, topic=?, text_content=?, annotation=?, 
            moderation_result=?, is_approved=?, media_descriptions=?,
            updated_at=CURRENT_TIMESTAMP
            WHERE id=?''', 
            (data['subject'], data['topic'], data['text_content'], 
             data['annotation'], data['moderation_result'], 
             data['is_approved'], json.dumps(data.get('media_descriptions', [])),
             existing_id))
        material_id = existing_id
    else:
        c.execute('''INSERT INTO materials 
            (url_hash, url, subject, topic, text_content, annotation, 
             moderation_result, is_approved, media_descriptions) 
            VALUES (?,?,?,?,?,?,?,?,?)''',
            (url_hash, data['url'], data['subject'], data['topic'], 
             data['text_content'], data['annotation'], 
             data['moderation_result'], data['is_approved'],
             json.dumps(data.get('media_descriptions', []))))
        material_id = c.lastrowid
    
    conn.commit()
    conn.close()
    return material_id

# ======================== АГЕНТ ЗАГРУЗКИ ========================
def download_file(url: str) -> Path:
    """Скачивание файла"""
    filename = DOWNLOAD_DIR / f"{get_url_hash(url)[:16]}_{Path(url).name}"
    if filename.exists():
        return filename
    
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()
    
    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    return filename

def extract_text_from_file(filepath: Path) -> str:
    """Извлечение текста из разных форматов"""
    ext = filepath.suffix.lower()
    
    if ext == '.pdf':
        with open(filepath, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            return "\n".join([page.extract_text() for page in reader.pages])
    
    elif ext == '.docx':
        doc = docx.Document(filepath)
        return "\n".join([para.text for para in doc.paragraphs])
    
    elif ext == '.pptx':
        prs = Presentation(filepath)
        text = []
        for slide in prs.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    text.append(shape.text)
        return "\n".join(text)
    
    elif ext == '.txt':
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    
    return ""

def extract_media_from_file(filepath: Path) -> List[Path]:
    """Извлечение медиа (изображения из PDF/PPTX)"""
    media_files = []
    ext = filepath.suffix.lower()
    
    # Простая заглушка - в реале нужен pdfplumber/python-pptx для извлечения
    # Для демо просто помечаем, что файл содержит медиа
    
    return media_files

# ======================== АГЕНТ АНАЛИЗА (LLM) ========================
def analyze_material(text: str) -> Dict:
    """Модерация + извлечение метаданных через Claude"""
    
    prompt = f"""Проанализируй учебный материал и верни JSON:

{{
  "subject": "название дисциплины",
  "topic": "тема материала",
  "annotation": "краткая аннотация 2-3 предложения",
  "moderation": {{
    "is_approved": true/false,
    "reason": "причина решения",
    "recommendations": "рекомендации по улучшению"
  }}
}}

Критерии модерации:
- Соответствие образовательным стандартам
- Отсутствие недостоверной информации
- Структурированность материала
- Наличие источников (желательно)

Текст материала:
{text[:8000]}
"""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result = json.loads(response.content[0].text)
    return result

# ======================== АГЕНТ МЕДИА (Vision) ========================
def describe_image(image_path: Path) -> str:
    """Описание изображения через Claude Vision"""
    
    with open(image_path, 'rb') as f:
        image_data = f.read()
    
    import base64
    image_base64 = base64.standard_b64encode(image_data).decode('utf-8')
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_base64
                    }
                },
                {
                    "type": "text",
                    "text": "Опиши это изображение из учебного материала: что изображено, какую учебную цель оно преследует."
                }
            ]
        }]
    )
    
    return response.content[0].text

def describe_video(video_path: Path) -> str:
    """Описание видео (анализ ключевых кадров)"""
    cap = cv2.VideoCapture(str(video_path))
    
    # Берем 3 кадра: начало, середина, конец
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_to_capture = [0, frame_count // 2, frame_count - 1]
    
    descriptions = []
    for frame_num in frames_to_capture:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if ret:
            frame_path = DOWNLOAD_DIR / f"frame_{frame_num}.jpg"
            cv2.imwrite(str(frame_path), frame)
            desc = describe_image(frame_path)
            descriptions.append(f"Кадр {frame_num}: {desc}")
    
    cap.release()
    return "\n".join(descriptions)

# ======================== АГЕНТ ЗАГРУЗКИ (основной) ========================
def load_material_agent(url: str) -> int:
    """Полный цикл загрузки и обработки материала"""
    
    print(f"📥 Загрузка: {url}")
    
    # 1. Проверка дубликатов
    existing_id = material_exists(url)
    if existing_id:
        print(f"✅ Материал уже существует (ID: {existing_id}), обновляем...")
    
    # 2. Скачивание
    filepath = download_file(url)
    print(f"💾 Сохранено: {filepath}")
    
    # 3. Извлечение текста
    text = extract_text_from_file(filepath)
    print(f"📝 Извлечено {len(text)} символов текста")
    
    # 4. Анализ через LLM
    analysis = analyze_material(text)
    print(f"🔍 Анализ завершен: {analysis['subject']} / {analysis['topic']}")
    
    # 5. Обработка медиа
    media_files = extract_media_from_file(filepath)
    media_descriptions = []
    
    for media_file in media_files:
        if media_file.suffix.lower() in ['.jpg', '.png', '.jpeg']:
            desc = describe_image(media_file)
            media_descriptions.append({"type": "image", "path": str(media_file), "description": desc})
        elif media_file.suffix.lower() in ['.mp4', '.avi']:
            desc = describe_video(media_file)
            media_descriptions.append({"type": "video", "path": str(media_file), "description": desc})
    
    # 6. Сохранение в БД
    material_data = {
        'url': url,
        'subject': analysis['subject'],
        'topic': analysis['topic'],
        'text_content': text,
        'annotation': analysis['annotation'],
        'moderation_result': analysis['moderation']['reason'],
        'is_approved': 1 if analysis['moderation']['is_approved'] else 0,
        'media_descriptions': media_descriptions
    }
    
    material_id = save_material(material_data)
    print(f"✅ Материал сохранен с ID: {material_id}")
    
    return material_id

# ======================== ФОРМИРОВАНИЕ ДАТАСЕТА ========================
def create_dataset() -> pd.DataFrame:
    """Экспорт данных из БД в DataFrame"""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT 
            id,
            subject,
            topic,
            annotation,
            LENGTH(text_content) as text_length,
            LENGTH(text_content) - LENGTH(REPLACE(text_content, ' ', '')) + 1 as word_count,
            is_approved,
            moderation_result,
            media_descriptions,
            prev_material_id,
            next_material_id,
            created_at
        FROM materials
    """, conn)
    conn.close()
    
    return df

# ======================== АНАЛИЗ ПРИЗНАКОВ ========================
def analyze_features(df: pd.DataFrame):
    """Анализ значимых признаков"""
    
    # 1. Корреляционный анализ числовых признаков
    numeric_cols = ['text_length', 'word_count', 'is_approved']
    correlation_matrix = df[numeric_cols].corr()
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
    plt.title('Корреляция признаков')
    plt.tight_layout()
    plt.savefig('correlation_matrix.png')
    plt.close()
    
    # 2. TF-IDF для анализа схожести тем
    vectorizer = TfidfVectorizer(max_features=100)
    tfidf_matrix = vectorizer.fit_transform(df['topic'].fillna(''))
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    # 3. Определение prev/next на основе схожести
    for idx, row in df.iterrows():
        similarities = similarity_matrix[idx]
        most_similar = similarities.argsort()[-3:-1][::-1]  # топ-2 схожих (кроме себя)
        
        # Простая логика: prev = более ранний похожий, next = более поздний
        candidates = df.iloc[most_similar]
        prev_candidate = candidates[candidates['created_at'] < row['created_at']]
        next_candidate = candidates[candidates['created_at'] > row['created_at']]
        
        if not prev_candidate.empty:
            update_material_links(row['id'], prev_id=prev_candidate.iloc[0]['id'])
        if not next_candidate.empty:
            update_material_links(row['id'], next_id=next_candidate.iloc[0]['id'])
    
    print("✅ Анализ признаков завершен")

def update_material_links(material_id: int, prev_id: int = None, next_id: int = None):
    """Обновление связей между материалами"""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    
    if prev_id:
        c.execute("UPDATE materials SET prev_material_id = ? WHERE id = ?", (prev_id, material_id))
    if next_id:
        c.execute("UPDATE materials SET next_material_id = ? WHERE id = ?", (next_id, material_id))
    
    conn.commit()
    conn.close()

# ======================== ОПИСАНИЕ СТРУКТУРЫ ========================
def describe_dataset_structure(df: pd.DataFrame):
    """Генерация описания структуры датасета"""
    
    report = []
    
    for col in df.columns:
        dtype = df[col].dtype
        unique_count = df[col].nunique()
        most_common = df[col].value_counts().head(3).to_dict() if df[col].dtype == 'object' else None
        
        report.append({
            'Признак': col,
            'Тип данных': str(dtype),
            'Описание': get_column_description(col),
            'Уникальных значений': unique_count,
            'Наиболее частые': most_common
        })
    
    # Текстовые характеристики
    text_stats = df[['text_length', 'word_count']].describe()
    
    # Визуализации
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    df['subject'].value_counts().plot(kind='bar', ax=axes[0, 0], title='Распределение по предметам')
    df['is_approved'].value_counts().plot(kind='pie', ax=axes[0, 1], title='Модерация', autopct='%1.1f%%')
    df['text_length'].hist(bins=30, ax=axes[1, 0], title='Распределение длины текста')
    df['word_count'].hist(bins=30, ax=axes[1, 1], title='Распределение количества слов')
    
    plt.tight_layout()
    plt.savefig('dataset_analysis.png')
    plt.close()
    
    # Сохранение отчета
    report_df = pd.DataFrame(report)
    report_df.to_csv('dataset_structure.csv', index=False, encoding='utf-8')
    
    with open('text_statistics.json', 'w', encoding='utf-8') as f:
        json.dump(text_stats.to_dict(), f, ensure_ascii=False, indent=2)
    
    print("✅ Описание структуры сохранено в dataset_structure.csv и text_statistics.json")

def get_column_description(col: str) -> str:
    """Описание назначения столбца"""
    descriptions = {
        'id': 'Уникальный идентификатор материала',
        'subject': 'Название дисциплины',
        'topic': 'Тема учебного материала',
        'annotation': 'Краткая аннотация содержания',
        'text_length': 'Длина текстового содержания в символах',
        'word_count': 'Количество слов в материале',
        'is_approved': 'Результат модерации (1 - одобрен, 0 - отклонен)',
        'moderation_result': 'Текстовое обоснование решения модератора',
        'media_descriptions': 'JSON-описания медиаконтента',
        'prev_material_id': 'ID предыдущего материала в последовательности изучения',
        'next_material_id': 'ID следующего материала в последовательности изучения',
        'created_at': 'Дата и время добавления материала'
    }
    return descriptions.get(col, 'Дополнительный признак')

# ======================== АГЕНТ ГЕНЕРАЦИИ ========================
def suggest_missing_topics(df: pd.DataFrame) -> List[str]:
    """Определение недостающих тем"""
    
    # Группировка по предметам
    subjects = df.groupby('subject')['topic'].apply(list).to_dict()
    
    suggestions = []
    
    for subject, topics in subjects.items():
        topics_text = "\n".join(topics)
        
        prompt = f"""Предмет: {subject}

Существующие темы:
{topics_text}

Проанализируй последовательность тем и предложи 3-5 недостающих тем, которые:
1. Логически заполняют пробелы между существующими темами
2. Обеспечивают последовательное изучение дисциплины
3. Соответствуют стандартной учебной программе

Верни только список тем (по одной на строку):"""
        
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        
        missing_topics = response.content[0].text.strip().split('\n')
        suggestions.extend([f"{subject}: {topic.strip('- ')}" for topic in missing_topics])
    
    return suggestions

def generate_material(subject: str, topic: str) -> Dict:
    """Генерация нового учебного материала"""
    
    prompt = f"""Создай учебный материал по теме:
Предмет: {subject}
Тема: {topic}

Требования:
- Структура: введение, основное содержание, заключение, контрольные вопросы
- Объем: 2000-3000 символов
- Научный стиль
- Примеры и иллюстрации (опиши словами, что должно быть изображено)

Верни JSON:
{{
  "text_content": "полный текст материала",
  "annotation": "краткая аннотация",
  "suggested_images": ["описание 1", "описание 2"]
}}"""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=4000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    generated = json.loads(response.content[0].text)
    generated['subject'] = subject
    generated['topic'] = topic
    
    return generated

def expand_dataset_agent():
    """Расширение датасета новыми материалами"""
    
    df = create_dataset()
    suggestions = suggest_missing_topics(df)
    
    print("\n🔍 Предложения по расширению датасета:\n")
    for i, suggestion in enumerate(suggestions, 1):
        print(f"{i}. {suggestion}")
    
    print("\nВведите номера тем для генерации (через запятую) или 'all' для всех:")
    user_input = input("> ")
    
    if user_input.lower() == 'all':
        selected = suggestions
    else:
        indices = [int(x.strip()) - 1 for x in user_input.split(',')]
        selected = [suggestions[i] for i in indices]
    
    for suggestion in selected:
        subject, topic = suggestion.split(': ', 1)
        print(f"\n📝 Генерация: {topic}")
        
        generated = generate_material(subject, topic)
        
        # Модерация сгенерированного
        analysis = analyze_material(generated['text_content'])
        
        # Сохранение
        material_data = {
            'url': f"generated://{get_url_hash(topic)}",
            'subject': subject,
            'topic': topic,
            'text_content': generated['text_content'],
            'annotation': generated['annotation'],
            'moderation_result': analysis['moderation']['reason'],
            'is_approved': 1 if analysis['moderation']['is_approved'] else 0,
            'media_descriptions': [{"type": "suggested_image", "description": img} 
                                   for img in generated.get('suggested_images', [])]
        }
        
        material_id = save_material(material_data)
        print(f"✅ Сгенерирован материал ID: {material_id}")

# ======================== ОТЧЕТ ========================
def generate_report():
    """Генерация финального отчета"""
    
    df = create_dataset()
    
    report = f"""
# ОТЧЕТ ПО РАБОТЕ АГЕНТНОЙ СИСТЕМЫ

## 1. Статистика базы данных
- Всего материалов: {len(df)}
- Одобрено модерацией: {df['is_approved'].sum()} ({df['is_approved'].mean()*100:.1f}%)
- Уникальных предметов: {df['subject'].nunique()}
- Уникальных тем: {df['topic'].nunique()}

## 2. Структура данных
Файлы:
- dataset_structure.csv - описание всех признаков
- text_statistics.json - статистика текстовых полей
- dataset_analysis.png - визуализации распределений
- correlation_matrix.png - корреляция признаков

## 3. Реализованный функционал

### Агент загрузки
- Поддержка форматов: PDF, DOCX, PPTX, TXT
- Дедупликация по хешу URL
- Автоматическое обновление при повторном запуске

### Агент анализа
- Извлечение метаданных (предмет, тема, аннотация)
- Модерация на соответствие методическим рекомендациям
- Критерии: достоверность, структурированность, соответствие стандартам

### Агент медиа
- Описание изображений через Claude Vision
- Анализ видео по ключевым кадрам
- Интеграция описаний в текстовый датасет

### Агент генерации
- Анализ пробелов в датасете
- Генерация материалов по недостающим темам
- Автоматическая модерация сгенерированного контента

## 4. Формат итоговых файлов

### База данных (SQLite)
Таблица materials:
- id, url, subject, topic, text_content, annotation
- moderation_result, is_approved
- media_descriptions (JSON)
- prev_material_id, next_material_id
- created_at, updated_at

### Экспорт датасета
CSV с полями:
- id, subject, topic, annotation
- text_length, word_count
- is_approved, moderation_result
- prev/next материалы

## 5. Примеры записей
{df.head(3).to_dict('records')}

## 6. Технологический стек
- Python 3.10+
- Claude API (Sonnet 4) для анализа и генерации
- SQLite для хранения
- scikit-learn для анализа признаков
- pandas, matplotlib, seaborn для визуализации

Дата формирования отчета: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
    
    with open('final_report.md', 'w', encoding='utf-8') as f:
        f.write(report)
    
    print("✅ Отчет сохранен в final_report.md")

# ======================== ГЛАВНОЕ МЕНЮ ========================
def main():
    """Главная точка входа"""
    
    init_db()
    
    print("""
╔═══════════════════════════════════════════════════════════╗
║   АГЕНТНАЯ СИСТЕМА ДЛЯ РАБОТЫ С УЧЕБНЫМИ МАТЕРИАЛАМИ     ║
╚═══════════════════════════════════════════════════════════╝

Выберите действие:
1. Загрузить материалы по списку URL
2. Сформировать датасет
3. Анализ признаков и связей
4. Описание структуры датасета
5. Расширение датасета (генерация)
6. Сгенерировать итоговый отчет
0. Выход
""")
    
    while True:
        choice = input("\n> Введите номер: ").strip()
        
        if choice == '1':
            print("\nВведите URL материалов (по одному на строку, пустая строка для завершения):")
            urls = []
            while True:
                url = input("URL: ").strip()
                if not url:
                    break
                urls.append(url)
            
            for url in urls:
                try:
                    load_material_agent(url)
                except Exception as e:
                    print(f"❌ Ошибка при обработке {url}: {e}")
        
        elif choice == '2':
            df = create_dataset()
            df.to_csv('educational_dataset.csv', index=False, encoding='utf-8')
            print(f"✅ Датасет сформирован: {len(df)} записей → educational_dataset.csv")
        
        elif choice == '3':
            df = create_dataset()
            analyze_features(df)
        
        elif choice == '4':
            df = create_dataset()
            describe_dataset_structure(df)
        
        elif choice == '5':
            expand_dataset_agent()
        
        elif choice == '6':
            generate_report()
        
        elif choice == '0':
            print("До свидания!")
            break
        
        else:
            print("❌ Неверный выбор")

if __name__ == "__main__":
    main()